# Notebook 04 — Feature Engineering
**Traceability:** Issue #4 · Depends on `03_eda.ipynb` outputs

---

## Objective

Construct the final **2D feature matrices** consumed by downstream modeling notebooks. Five dataset variants (DS-A through DS-E) are produced with precisely-controlled outlier treatment, scaling, and temporal feature depth.

---

## Inputs and Dependencies

| Input | Path | Produced by |
|-------|------|-------------|
| Cleaned training data | `../data/processed/train_cleaned.csv` | `02_data_cleaning.ipynb` |
| Cleaned test data source | `../data/processed/valid_cleaned.csv` | `02_data_cleaning.ipynb` |

**Required columns:** `unit_number`, `time_cycles`, `RUL`, `s_2 … s_21` (14 retained sensors)

---

## Methodology

All operations follow strict **train-fit / test-apply** discipline — no scaler, PCA, or winsorization bounds are computed from test data.

Based on EDA analysis, five targeted dataset variants are produced:

| Dataset | Outlier Treatment | Feature Set | Scaling | PCA | Primary Use |
|---------|-------------------|-------------|---------|-----|-------------|
| **DS-A** | None | Raw sensors + clipped RUL | MinMax | No | Baseline |
| **DS-B** | Winsorize 1–99% | Lag + Rolling + Diff + EWMA + HI | MinMax | No | Classical ML (trees, ensembles) |
| **DS-C** | Winsorize 1–99% | Lag + Rolling + Diff + EWMA | MinMax | 95% var | Linear / regularized models |
| **DS-D** | None | Rolling + Diff + HI | MinMax | No | Deep Learning (LSTM, Transformer) |
| **DS-E** | Clip 1–99% | Lag + Rolling + Diff + EWMA + HI | MinMax | No | Full comprehensive / stacking |

**Key EDA decisions encoded here:**
- `s_9` and `s_14` outliers (~8%) are failure-relevant — no aggressive clipping for DL (DS-D).
- Non-linear late-stage acceleration → diff and rolling features are essential.
- PC1 explains ~69.6% variance → health index is a strong composite feature.
- 14 correlated sensors → PCA reduction justified for compact models (DS-C).

---

## Outputs Produced

- `../data/experiments/DS-{A,B,C,D,E}/train.csv` — training split (2D)
- `../data/experiments/DS-{A,B,C,D,E}/test.csv` — held-out testing split (2D)

In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

# ── Paths ─────────────────────────────────────────────────────────────
DATA_DIR   = Path("../data/processed")
RAW_DIR    = Path("../data")
EXP_DIR    = Path("../data/experiments")
TRAIN_PATH = DATA_DIR / "train_cleaned.csv"
VALID_PATH = DATA_DIR / "valid_cleaned.csv"

# ── Constants ─────────────────────────────────────────────────────────
RUL_CLIP      = 125           # piecewise-linear RUL cap (Source A)
ROLL_WINDOWS  = [5, 10, 20]  # rolling statistics window sizes
LAG_STEPS     = [1, 2, 5]    # lag steps for temporal features
EWMA_SPANS    = [5, 10]      # EWMA span parameters
WINDOW_LENGTH = 30            # sequence window for DL (Source C)
SHIFT         = 1             # sequence step size
RANDOM_STATE  = 42

# ── Helpers ───────────────────────────────────────────────────────────
def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for col in ["unit_number", "time_cycles", "RUL"]:
        assert col in df.columns, f"Missing required column '{col}' in {path.name}"
    return df.sort_values(["unit_number", "time_cycles"]).reset_index(drop=True)


# ── 1. DATA POLICY (USER-REQUESTED) ───────────────────────────────────
# Use full train_cleaned.csv as train data.
# Use valid_cleaned.csv as test data.
# No validation split is used in modeling from this notebook.
df_train_raw = load_data(TRAIN_PATH)
df_test_raw  = load_data(VALID_PATH)

sensor_cols  = sorted([c for c in df_train_raw.columns if c.startswith("s_")],
                      key=lambda x: int(x.split("_")[1]))
setting_cols = [c for c in df_train_raw.columns if c.startswith("setting_")]

print(f"Train rows : {df_train_raw.shape[0]} (full train_cleaned.csv)")
print(f"Test  rows : {df_test_raw.shape[0]} (from valid_cleaned.csv)")
print(f"Sensors ({len(sensor_cols)}): {sensor_cols}")

Train rows : 20631 (full train_cleaned.csv)
Test  rows : 13096 (from valid_cleaned.csv)
Sensors (14): ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']


## 1 — Reusable Pipeline Components

All operations are defined as functions so each dataset variant can be assembled declaratively.

In [3]:
# ── 1a) RUL clipping ──────────────────────────────────────────────────
def clip_rul(df: pd.DataFrame, upper: int = RUL_CLIP) -> pd.DataFrame:
    df = df.copy()
    df["RUL_clipped"] = df["RUL"].clip(upper=upper)
    return df


# ── 1b) Outlier handling ─────────────────────────────────────────────
def winsorize_sensors(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    sensors: list[str],
    limits: tuple = (0.01, 0.01),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Winsorize: compute 1–99% bounds on train only, apply to test."""
    df_tr, df_te = df_train.copy(), df_test.copy()
    for s in sensors:
        lo = df_tr[s].quantile(limits[0])
        hi = df_tr[s].quantile(1 - limits[1])
        df_tr[s] = df_tr[s].clip(lo, hi)
        df_te[s] = df_te[s].clip(lo, hi)
    return df_tr, df_te


def clip_sensors(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    sensors: list[str],
    lower_q: float = 0.01,
    upper_q: float = 0.99,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Percentile clip: fit bounds on train only, apply to test."""
    df_tr, df_te = df_train.copy(), df_test.copy()
    for s in sensors:
        lo = df_tr[s].quantile(lower_q)
        hi = df_tr[s].quantile(upper_q)
        df_tr[s] = df_tr[s].clip(lo, hi)
        df_te[s] = df_te[s].clip(lo, hi)
    return df_tr, df_te


# ── 1c) Scaling ───────────────────────────────────────────────────────
def minmax_scale(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, MinMaxScaler]:
    """Fit scaler on train only; transform test with same scaler."""
    df_tr, df_te = df_train.copy(), df_test.copy()
    scaler = MinMaxScaler()
    df_tr[cols] = scaler.fit_transform(df_tr[cols])
    df_te[cols] = scaler.transform(df_te[cols])
    return df_tr, df_te, scaler


# ── 1d) Temporal feature engineering ─────────────────────────────────
# All operations use groupby('unit_number') to prevent cross-engine leakage.

def add_lag_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)
    for s in sensors:
        for lag in LAG_STEPS:
            out[f"{s}_lag_{lag}"] = grouped[s].shift(lag)
    return out


def add_rolling_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)
    for s in sensors:
        for w in ROLL_WINDOWS:
            out[f"{s}_rmean_{w}"] = grouped[s].transform(
                lambda x: x.rolling(w, min_periods=1).mean()
            )
            out[f"{s}_rstd_{w}"] = grouped[s].transform(
                lambda x: x.rolling(w, min_periods=1).std().fillna(0)
            )
    return out


def add_diff_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)
    for s in sensors:
        out[f"{s}_diff1"] = grouped[s].diff().fillna(0)
    return out


def add_ewma_features(df: pd.DataFrame, sensors: list[str]) -> pd.DataFrame:
    out = df.copy()
    grouped = out.groupby("unit_number", group_keys=False)
    for s in sensors:
        for span in EWMA_SPANS:
            out[f"{s}_ewma_{span}"] = grouped[s].transform(
                lambda x, sp=span: x.ewm(span=sp, min_periods=1).mean()
            )
    return out


def add_health_index(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    sensors: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    PC1-based health index: higher value = healthier engine.

    Leakage-safe: PCA fit on train only; MinMaxScaler fit on train hi only,
    then applied to test.
    """
    df_tr, df_te = df_train.copy(), df_test.copy()
    pca = PCA(n_components=1, random_state=RANDOM_STATE)

    hi_train = pca.fit_transform(df_tr[sensors]).ravel()
    hi_test  = pca.transform(df_te[sensors]).ravel()

    # Fit scaler on train hi only — no leakage
    hi_scaler = MinMaxScaler()
    hi_scaler.fit(hi_train.reshape(-1, 1))

    df_tr["health_index"] = 1.0 - hi_scaler.transform(hi_train.reshape(-1, 1)).ravel()
    df_te["health_index"] = 1.0 - hi_scaler.transform(hi_test.reshape(-1, 1)).ravel()
    return df_tr, df_te


# ── 1e) PCA dimensionality reduction ─────────────────────────────────
def apply_pca(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    feature_cols: list[str],
    variance_threshold: float = 0.95,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Replace feature_cols with PCA components retaining given variance. Fit on train only."""
    pca = PCA(n_components=variance_threshold, random_state=RANDOM_STATE)

    tr_pca = pca.fit_transform(df_train[feature_cols])
    te_pca = pca.transform(df_test[feature_cols])

    n_comp   = tr_pca.shape[1]
    pc_names = [f"PC_{i+1}" for i in range(n_comp)]

    df_tr = df_train.drop(columns=feature_cols).copy()
    df_te = df_test.drop(columns=feature_cols).copy()

    for i, name in enumerate(pc_names):
        df_tr[name] = tr_pca[:, i]
        df_te[name] = te_pca[:, i]

    print(f"  PCA: {len(feature_cols)} features → {n_comp} components ({variance_threshold*100:.0f}% variance)")
    return df_tr, df_te, pc_names


# ── 1f) Cleanup and export ────────────────────────────────────────────
def finalize_and_save(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    dataset_name: str,
    drop_meta: bool = True,
) -> None:
    """Fill lag-generated NaNs with 0, validate schema, and save train/test CSVs."""
    df_train = df_train.fillna(0)
    df_test  = df_test.fillna(0)

    out_dir = EXP_DIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    meta_cols = ["time_cycles", "RUL"] + setting_cols
    splits = {"train": df_train, "test": df_test}

    for split_name, df in splits.items():
        if drop_meta:
            keep = [c for c in df.columns if c not in meta_cols]
        else:
            keep = list(df.columns)
        df[keep].to_csv(out_dir / f"{split_name}.csv", index=False)

    keep_tr  = [c for c in df_train.columns if c not in meta_cols] if drop_meta else list(df_train.columns)
    tr_nan   = int(df_train[keep_tr].isna().sum().sum())
    te_nan   = int(df_test[keep_tr].isna().sum().sum())

    print(f"  ✅ {dataset_name}: train {df_train[keep_tr].shape}, test {df_test[keep_tr].shape}")
    print(f"     NaN: train={tr_nan}, test={te_nan}")
    assert "RUL_clipped" in keep_tr, f"{dataset_name}: RUL_clipped missing from output"


print("✅ Pipeline components defined")

✅ Pipeline components defined


## 2 — DS-A: Baseline (raw sensors, MinMax scaled, clipped RUL)

**Rationale:** Minimal processing. Establishes the floor performance that all other variants must beat. No outlier treatment, no engineered features — just scaled sensors and piecewise RUL.

In [4]:
print("═" * 60)
print("DS-A: Baseline")
print("═" * 60)

tr_a = clip_rul(df_train_raw)
te_a = clip_rul(df_test_raw)

tr_a, te_a, _ = minmax_scale(tr_a, te_a, sensor_cols)

finalize_and_save(tr_a, te_a, "DS-A")

════════════════════════════════════════════════════════════
DS-A: Baseline
════════════════════════════════════════════════════════════
  ✅ DS-A: train (20631, 16), test (13096, 16)
     NaN: train=0, test=0


## 3 — DS-B: Classical ML (Winsorize + Full features + Health Index)

**Rationale:** The EDA showed non-linear late-stage trends, so lag, rolling, diff and EWMA features capture rate-of-change and local trajectory shape. Winsorizing at 1–99% compresses the `s_9`/`s_14` tails gently without destroying failure-relevant signal. Health index adds a powerful PCA-derived composite. Designed for Random Forest, XGBoost, and similar tree/ensemble models.

In [5]:
print("═" * 60)
print("DS-B: Classical ML (Winsorize + Full features + HI)")
print("═" * 60)

tr_b = clip_rul(df_train_raw)
te_b = clip_rul(df_test_raw)

# Outlier: winsorize
tr_b, te_b = winsorize_sensors(tr_b, te_b, sensor_cols)

# Scale sensors
tr_b, te_b, _ = minmax_scale(tr_b, te_b, sensor_cols)

# Health index (on scaled sensors)
tr_b, te_b = add_health_index(tr_b, te_b, sensor_cols)

# Feature engineering
tr_b = add_lag_features(tr_b, sensor_cols)
tr_b = add_rolling_features(tr_b, sensor_cols)
tr_b = add_diff_features(tr_b, sensor_cols)
tr_b = add_ewma_features(tr_b, sensor_cols)

te_b = add_lag_features(te_b, sensor_cols)
te_b = add_rolling_features(te_b, sensor_cols)
te_b = add_diff_features(te_b, sensor_cols)
te_b = add_ewma_features(te_b, sensor_cols)

finalize_and_save(tr_b, te_b, "DS-B")

════════════════════════════════════════════════════════════
DS-B: Classical ML (Winsorize + Full features + HI)
════════════════════════════════════════════════════════════
  ✅ DS-B: train (20631, 185), test (13096, 185)
     NaN: train=0, test=0


## 4 — DS-C: Compact ML (Winsorize + Full features + PCA 95%)

**Rationale:** Same feature engineering as DS-B, but the high dimensionality (~300+ features) is compressed via PCA retaining 95% variance. The correlation heatmap showed strong redundancy, and PCA at 95% preserves signal while eliminating collinearity. Designed for linear models, SVR, and regularized regression.

In [6]:
print("═" * 60)
print("DS-C: Compact ML (Winsorize + Full features + PCA 95%)")
print("═" * 60)

tr_c = clip_rul(df_train_raw)
te_c = clip_rul(df_test_raw)

# Outlier: winsorize
tr_c, te_c = winsorize_sensors(tr_c, te_c, sensor_cols)

# Scale sensors
tr_c, te_c, _ = minmax_scale(tr_c, te_c, sensor_cols)

# Feature engineering (same as DS-B but no health_index — PCA replaces it)
tr_c = add_lag_features(tr_c, sensor_cols)
tr_c = add_rolling_features(tr_c, sensor_cols)
tr_c = add_diff_features(tr_c, sensor_cols)
tr_c = add_ewma_features(tr_c, sensor_cols)

te_c = add_lag_features(te_c, sensor_cols)
te_c = add_rolling_features(te_c, sensor_cols)
te_c = add_diff_features(te_c, sensor_cols)
te_c = add_ewma_features(te_c, sensor_cols)

# Fill NaN before PCA (lag-created NaNs)
tr_c = tr_c.fillna(0)
te_c = te_c.fillna(0)

# PCA on all engineered features (excluding meta and target)
meta_cols_pca = {"unit_number", "time_cycles", "RUL", "RUL_clipped"} | set(setting_cols)
feat_cols_c = [c for c in tr_c.columns if c not in meta_cols_pca]

tr_c, te_c, pc_names = apply_pca(tr_c, te_c, feat_cols_c, variance_threshold=0.95)

finalize_and_save(tr_c, te_c, "DS-C")

════════════════════════════════════════════════════════════
DS-C: Compact ML (Winsorize + Full features + PCA 95%)
════════════════════════════════════════════════════════════
  PCA: 182 features → 33 components (95% variance)
  ✅ DS-C: train (20631, 35), test (13096, 35)
     NaN: train=0, test=0


## 5 — DS-D: Deep Learning (Raw sensors + Rolling + Diff + Health Index)

**Rationale:** Sequence models (LSTM, Transformer) learn temporal patterns internally, so heavy feature engineering adds noise rather than signal. We keep: (1) raw scaled sensors to preserve fine-grained interactions, (2) rolling mean/std to provide local context, (3) diff to highlight rate-of-change, and (4) health index as a single degradation summary. No outlier clipping — the EDA showed `s_9`/`s_14` extremes are failure-relevant and DL models handle them well.

In [7]:
print("═" * 60)
print("DS-D: Deep Learning (Raw + Rolling + Diff + HI)")
print("═" * 60)

tr_d = clip_rul(df_train_raw)
te_d = clip_rul(df_test_raw)

# No outlier treatment — preserve extremes for sequence models
# Scale sensors
tr_d, te_d, _ = minmax_scale(tr_d, te_d, sensor_cols)

# Health index
tr_d, te_d = add_health_index(tr_d, te_d, sensor_cols)

# Lighter features: rolling + diff only (no lag, no EWMA — model learns temporal)
tr_d = add_rolling_features(tr_d, sensor_cols)
tr_d = add_diff_features(tr_d, sensor_cols)

te_d = add_rolling_features(te_d, sensor_cols)
te_d = add_diff_features(te_d, sensor_cols)

finalize_and_save(tr_d, te_d, "DS-D")

════════════════════════════════════════════════════════════
DS-D: Deep Learning (Raw + Rolling + Diff + HI)
════════════════════════════════════════════════════════════
  ✅ DS-D: train (20631, 115), test (13096, 115)
     NaN: train=0, test=0


## 6 — DS-E: Extended Temporal (All Features)

**Rationale:** Maximum temporal context for attention-based and recurrent models. Every enrichment layer — lag features, rolling statistics, first-order differences, and EWMA — is stacked on top of the winsorized, scaled sensor signals. Intended for experiments measuring the benefit of temporal richness vs. dimensionality cost.

| Treatment | Applied |
|-----------|---------|  
| RUL clip | 125 cycles |
| Outlier treatment | Winsorize (1–99%) on sensors |
| Scaling | MinMax on sensors (fit on train only) |
| Health index | PC1-based (fit on train only) |
| Lag features | Steps 1, 2, 5 |
| Rolling stats | Windows 5, 10, 20 (mean + std) |
| First differences | Δ1 per sensor |
| EWMA | Spans 5, 10 |


In [8]:
print("═" * 60)
print("DS-E: Extended Temporal (All Features)")
print("═" * 60)

tr_e = clip_rul(df_train_raw)
te_e = clip_rul(df_test_raw)

# Winsorize before temporal expansion to reduce outlier influence
tr_e, te_e = winsorize_sensors(tr_e, te_e, sensor_cols)

# Scale sensors (fit on train only)
tr_e, te_e, _ = minmax_scale(tr_e, te_e, sensor_cols)

# Health index (leakage-safe: PCA + MinMax fit on train only)
tr_e, te_e = add_health_index(tr_e, te_e, sensor_cols)

# All temporal enrichment layers (applied per split, no cross-leakage)
tr_e = add_lag_features(tr_e, sensor_cols)
tr_e = add_rolling_features(tr_e, sensor_cols)
tr_e = add_diff_features(tr_e, sensor_cols)
tr_e = add_ewma_features(tr_e, sensor_cols)

te_e = add_lag_features(te_e, sensor_cols)
te_e = add_rolling_features(te_e, sensor_cols)
te_e = add_diff_features(te_e, sensor_cols)
te_e = add_ewma_features(te_e, sensor_cols)

finalize_and_save(tr_e, te_e, "DS-E")

════════════════════════════════════════════════════════════
DS-E: Extended Temporal (All Features)
════════════════════════════════════════════════════════════
  ✅ DS-E: train (20631, 185), test (13096, 185)
     NaN: train=0, test=0


## Dataset Summary and Validation

Validate all five dataset variants: row counts, feature counts, NaN counts, and presence of the target column for **train/test** outputs.

## Feature Dictionary and Lineage

Every feature in the output datasets is traceable to an EDA finding or design decision.

### Base Input Features (14 sensors)

| Feature | Sensor Description | EDA Priority | RUL Correlation |
|---------|--------------------|-------------|-----------------|
| `s_2` | Total temperature at fan inlet (LPC) | High | — |
| `s_3` | Total temperature at LPC outlet | High | — |
| `s_4` | Total temperature at HPC outlet | High | −0.68 |
| `s_7` | Total pressure at fan inlet | High | +0.66 |
| `s_8` | Total pressure at LPC outlet | Medium | — |
| `s_9` | Total pressure at HPC outlet | High | — |
| `s_11` | Static pressure at HPC outlet | High | −0.70 |
| `s_12` | Ratio of fuel flow to PS30 | High | +0.67 |
| `s_13` | Corrected core speed | Medium | — |
| `s_14` | Corrected fan speed | High | — |
| `s_15` | Bypass duct pressure (HPT coolant bleed) | Medium | — |
| `s_17` | Bleed enthalpy | Medium | — |
| `s_20` | High-pressure turbine coolant bleed | Medium | — |
| `s_21` | Low-pressure turbine coolant bleed | Medium | — |

### Engineered Feature Types

| Feature Pattern | Formula | Purpose | EDA Motivation |
|----------------|---------|---------|----------------|
| `{s}_lag_{k}` | `groupby(unit).shift(k)` | Capture prior state at k=1,2,5 steps | Non-linear late-stage acceleration |
| `{s}_rmean_{w}` | `groupby(unit).rolling(w).mean()` | Smooth noise, reveal trend | Sensor noise suppression |
| `{s}_rstd_{w}` | `groupby(unit).rolling(w).std()` | Measure local volatility | Degradation often increases variance |
| `{s}_diff1` | `groupby(unit).diff()` | Rate-of-change per cycle | Sudden shifts are failure precursors |
| `{s}_ewma_{s}` | `groupby(unit).ewm(span=s).mean()` | Exponentially weighted smoothing | Prioritises recent cycles |
| `health_index` | `1 - MinMax(PC1)` fitted on train | Composite degradation score | PC1 explains ~69.6% of sensor variance |
| `PC_{n}` (DS-C) | PCA retaining 95% variance | Compact representation | 14 correlated sensors → redundancy |

### Target Variable

| Column | Definition | Used by |
|--------|-----------|---------|
| `RUL_clipped` | `min(RUL, 125)` — piecewise-linear RUL cap | All datasets (model target) |

**Rationale for clipping at 125:** Early in engine life, the linear RUL assumption is unreliable and predictive models should not be penalised for imprecisely estimating large RUL values (Source A; confirmed by EDA showing median lifetime ~206 cycles, right-skewed). The model focuses on the last 125 cycles where degradation signal is meaningful.

In [9]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

summary_rows = []
for ds_name in ["DS-A", "DS-B", "DS-C", "DS-D", "DS-E"]:
    ds_dir = EXP_DIR / ds_name
    tr = pd.read_csv(ds_dir / "train.csv")
    te = pd.read_csv(ds_dir / "test.csv")
    summary_rows.append({
        "Dataset": ds_name,
        "Train Rows": tr.shape[0],
        "Test Rows": te.shape[0],
        "Features": tr.shape[1] - 1,  # exclude RUL_clipped
        "Train NaN": int(tr.isna().sum().sum()),
        "Test NaN": int(te.isna().sum().sum()),
        "Has RUL_clipped": "RUL_clipped" in tr.columns,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print()

# Show feature column names for each dataset
for ds_name in ["DS-A", "DS-B", "DS-C", "DS-D", "DS-E"]:
    ds_dir = EXP_DIR / ds_name
    tr = pd.read_csv(ds_dir / "train.csv", nrows=1)
    feat_cols = [c for c in tr.columns if c != "RUL_clipped"]
    print(f"\n{ds_name} features ({len(feat_cols)}):")
    print(f"  {feat_cols[:10]}{'...' if len(feat_cols) > 10 else ''}")

print("\n✅ All datasets validated and ready for modeling")

DATASET SUMMARY
Dataset  Train Rows  Test Rows  Features  Train NaN  Test NaN  Has RUL_clipped
   DS-A       20631      13096        15          0         0             True
   DS-B       20631      13096       184          0         0             True
   DS-C       20631      13096        34          0         0             True
   DS-D       20631      13096       114          0         0             True
   DS-E       20631      13096       184          0         0             True


DS-A features (15):
  ['unit_number', 's_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13']...

DS-B features (184):
  ['unit_number', 's_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13']...

DS-C features (34):
  ['unit_number', 'PC_1', 'PC_2', 'PC_3', 'PC_4', 'PC_5', 'PC_6', 'PC_7', 'PC_8', 'PC_9']...

DS-D features (114):
  ['unit_number', 's_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13']...

DS-E features (184):
  ['unit_number', 's_2', 's_3', 's_4', 's_7', '

## Quality Gate C — Feature Integrity Gate

Before proceeding to modeling, verify all checklist items below.

| # | Check | Expected |
|---|-------|----------|
| C1 | Feature definitions stable and versioned | All 5 dataset directories written to `data/experiments/` |
| C2 | Train-fit / test-apply strictly enforced | MinMaxScaler, PCA, winsorization bounds fitted on train only |
| C3 | No cross-engine leakage in rolling/lagged features | All temporal ops use `groupby('unit_number')` |
| C4 | RUL target present in all output files | `RUL_clipped` column exists in every `train/test.csv` |
| C5 | No NaN in output files | `finalize_and_save` fills lag-generated NaNs with 0 and reports count |
| C6 | Health index leakage-safe | PCA and MinMaxScaler for `health_index` fitted on train hi only |
| C7 | DS-C PCA fit on train only | `apply_pca` fits on train, transforms test |
| C8 | Feature dictionary present | Lineage table above documents all features |

**Sign-off condition:** All checks pass before running `05_modeling_validation.ipynb` or `05_pytorch_deep_learning.ipynb`.

## Transition to Downstream Notebooks

### What was produced here

| Artifact | Path | Consumer |
|---------|------|---------|
| DS-A: Baseline flat CSV | `data/experiments/DS-A/{train,test}.csv` | `05_modeling_validation.ipynb` (baseline models) |
| DS-B: Classical ML flat CSV | `data/experiments/DS-B/{train,test}.csv` | `05_modeling_validation.ipynb` (tree/ensemble models) |
| DS-C: Compact ML flat CSV | `data/experiments/DS-C/{train,test}.csv` | `05_modeling_validation.ipynb` (linear/SVR models) |
| DS-D: DL flat CSV | `data/experiments/DS-D/{train,test}.csv` | `05_pytorch_deep_learning.ipynb` |
| DS-E: Full robust flat CSV | `data/experiments/DS-E/{train,test}.csv` | `05_modeling_validation.ipynb` (ensemble/ablation) |

### What the next notebooks consume

**`05_modeling_validation.ipynb`** (classical ML branch):
- Loads DS-A, DS-B, DS-C, DS-E flat CSVs.
- Target column: `RUL_clipped`.

**`05_pytorch_deep_learning.ipynb`** (deep learning branch):
- Loads DS-D flat CSV and builds 3D sequence tensors inside Notebook 05.
- Sequence shape: `(samples, 30, n_features)`.

### Assumptions carried forward

- `RUL_clipped` (cap=125) is the universal prediction target across all models.
- All scalers were fit on train split only — test is never seen during fitting.
- This notebook exports **2D tabular train/test data only**; no validation split is exported.